# 02 — ARC environment and public observations

**Status: UNEXECUTED FIRST DRAFT.** Purpose: wrap the official `arc-agi==0.9.9` toolkit with legal action validation and an honest environment recovery boundary. Inputs: notebook 00's configuration, notebook 01's recovery gate, and user-provided public environment assets in `environment_files/`. Outputs: public game catalogue and a bounded environment smoke report. Expected runtime: public asset preparation is bounded to ten minutes, then smoke takes seconds to minutes; profile on this host. Run All prepares missing public assets through the official SDK and performs one public OFFLINE reset and, when possible, one action; it never trains or submits.

Checkpoint directory: `checkpoints/<run_id>/` for callers; environment observations are **not** hidden game-state checkpoints. Exact resume: optimizer/data recovery comes from notebook 01. An interrupted ARC episode cannot be restored exactly through the verified public toolkit; restart it under a new episode ID and do not claim identical continuation. Asset preparation has a 600-second budget with at most 32 public games, and the smoke has a 30-second budget and two API actions. No acquisition or smoke is run while authoring. The official SDK may download and load third-party public game assets as runtime dependencies; project code never reads their source or private state.

Sources verified 2026-10-09: [Arcade API](https://docs.arcprize.org/toolkit/arc_agi), [EnvironmentWrapper API](https://docs.arcprize.org/toolkit/environment_wrapper), [official toolkit](https://github.com/arcprize/ARC-AGI). The [Kaggle data page](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3/data) was identified, but its runner protocol was not readable through the research interface. Generic toolkit operation is implemented; Kaggle submission packaging is not claimed or executed.

## Load definitions without executing dependencies
Only tagged imports and definitions are loaded. No dependency's training or data cells run here.

In [ ]:
from pathlib import Path
import json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks' / '00_setup_and_hardware.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Open a kernel in /workspace/arc-agi-3-ahrv2 or its notebooks directory.')
_nb = json.loads((ROOT / 'notebooks' / '00_setup_and_hardware.ipynb').read_text())
for _cell in _nb['cells']:
    if _cell['cell_type'] == 'code' and 'ahr-export' in _cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_cell['source']), '00_setup_and_hardware.ipynb', 'exec'), globals())
load_exports([1], globals())

## Actions and observation validation
ACTION6 always carries explicitly selected integer coordinates in `0..63`. The final returned animation frame is the model's current board; the other returned frames remain available as public animation data. No padding, clipping, guessed controls, or private state is substituted.

In [ ]:
from dataclasses import dataclass
from typing import Optional
import copy
import importlib.metadata
import logging
import time
import uuid
import numpy as np

@dataclass(frozen=True)
class ActionSpec:
    action: int
    x: Optional[int] = None
    y: Optional[int] = None

    def __post_init__(self):
        if isinstance(self.action, bool) or not isinstance(self.action, (int, np.integer)) or not 0 <= int(self.action) <= 7:
            raise ValueError('action must be an integer in 0..7 (0=RESET).')
        if int(self.action) == 6:
            for name, value in (('x', self.x), ('y', self.y)):
                if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or not 0 <= int(value) < 64:
                    raise ValueError(f'ACTION6 requires integer {name} in 0..63.')
        elif self.x is not None or self.y is not None:
            raise ValueError('Coordinates are only valid for ACTION6.')

    def to_dict(self):
        return {'action': int(self.action), 'x': None if self.x is None else int(self.x), 'y': None if self.y is None else int(self.y)}

    @classmethod
    def from_dict(cls, data):
        if not isinstance(data, dict):
            raise TypeError('An action record must be a dictionary.')
        return cls(data['action'], data.get('x'), data.get('y'))

def _arc_action_id(value):
    value = getattr(value, 'value', value)
    if isinstance(value, str):
        value = 0 if value.upper() == 'RESET' else value.upper().removeprefix('ACTION')
    if isinstance(value, bool):
        raise ValueError('Boolean action IDs are invalid.')
    result = int(value)
    if not 0 <= result <= 7:
        raise ValueError(f'Unsupported public action ID: {result}')
    return result

def validate_grid(grid):
    result = np.asarray(grid)
    if result.shape != (64, 64):
        raise ValueError(f'Expected a 64x64 public board, received {result.shape}.')
    if not np.issubdtype(result.dtype, np.integer) or np.any(result < 0) or np.any(result > 15):
        raise ValueError('Board values must be palette integers in 0..15.')
    return result.astype(np.int64, copy=True)

def validate_available_actions(values):
    actions = [_arc_action_id(value) for value in values]
    if len(actions) != len(set(actions)):
        raise ValueError('Duplicate available actions in observation.')
    return actions

def public_observation(frame_data, game_id=None):
    if frame_data is None:
        raise RuntimeError('The official toolkit returned no observation; no transition was recorded.')
    frames = getattr(frame_data, 'frame', None)
    if frames is None or len(frames) == 0:
        raise RuntimeError('The official response contains no public board.')
    array = np.asarray(frames)
    frame_list = [validate_grid(array)] if array.ndim == 2 else [validate_grid(frame) for frame in frames]
    state = getattr(frame_data, 'state', None)
    state = getattr(state, 'name', str(state))
    result = {
        'grid': frame_list[-1], 'frames': frame_list,
        'available_actions': validate_available_actions(getattr(frame_data, 'available_actions', [])),
        'levels_completed': int(getattr(frame_data, 'levels_completed', 0)),
        'state': state, 'game_id': str(getattr(frame_data, 'game_id', None) or game_id or ''),
        'guid': getattr(frame_data, 'guid', None),
        'full_reset': bool(getattr(frame_data, 'full_reset', False)),
        'provenance': {'kind': 'observed', 'interface': 'official-arc-agi-public-frame'},
    }
    if not result['game_id']:
        raise ValueError('Observation has no game_id.')
    return result

def observation_to_record(observation):
    return {key: ([frame.tolist() for frame in value] if key == 'frames' else value.tolist() if isinstance(value, np.ndarray) else copy.deepcopy(value))
            for key, value in observation.items()}

def terminal_observation(observation):
    return str(observation['state']).upper() in {'WIN', 'GAME_OVER'}

## Official session adapter
Only public catalogue metadata and returned observations cross the adapter boundary. OFFLINE requires pre-existing local public assets. ONLINE and COMPETITION make real API calls when explicitly configured; COMPETITION creates one scorecard, prohibits repeated game-family sessions in this adapter, and closes it once. No in-flight score is queried. RESET is a public API control action; it is not silently added to the observation's policy mask.

In [ ]:
class ARCSession:
    def __init__(self, cfg, operation_mode=None):
        import os
        from arc_agi import Arcade, OperationMode
        self.cfg = copy.deepcopy(cfg)
        settings = self.cfg.get('environment', {})
        self.operation_mode = str(operation_mode or settings.get('operation_mode', 'OFFLINE')).upper()
        if os.environ.get('OPERATION_MODE', '').strip().lower() == 'competition' and self.operation_mode != 'COMPETITION':
            raise RuntimeError('OPERATION_MODE=competition overrides the toolkit constructor; clear it before requesting development/OFFLINE operation.')
        if self.operation_mode not in {'OFFLINE', 'ONLINE', 'COMPETITION', 'NORMAL'}:
            raise ValueError('Unknown ARC operation mode.')
        # The official package may log anonymous API keys. Keep SDK logs out of notebook outputs.
        logger = logging.getLogger('ahr.arc.public_adapter')
        logger.handlers = [logging.NullHandler()]
        logger.propagate = False
        self.arcade = Arcade(
            operation_mode=getattr(OperationMode, self.operation_mode),
            environments_dir=str(ROOT / settings.get('environments_dir', 'environment_files')),
            recordings_dir=str(ROOT / settings.get('recordings_dir', 'data/toolkit_recordings')),
            logger=logger,
        )
        effective = str(getattr(self.arcade.operation_mode, 'value', self.arcade.operation_mode)).upper()
        if effective != self.operation_mode:
            raise RuntimeError(f'Effective toolkit mode {effective} differs from requested {self.operation_mode}; clear conflicting OPERATION_MODE and retry.')
        self.max_actions = int(settings.get('max_actions', 256))
        if self.max_actions <= 0:
            raise ValueError('max_actions must be positive.')
        self.env = None
        self.observation = None
        self.game_id = None
        self.episode_id = None
        self.actions_used = 0
        self.closed = False
        self.scorecard_id = None
        self._started_families = set()
        self.toolkit_version = importlib.metadata.version('arc-agi')

    def list_games(self):
        return sorted({str(info.game_id) for info in self.arcade.get_environments()})

    def reset(self, game_id):
        if self.closed:
            raise RuntimeError('Session is closed.')
        family = str(game_id).split('-', 1)[0]
        if self.operation_mode == 'COMPETITION' and family in self._started_families:
            raise RuntimeError('Competition adapter permits one game session per family; open a separately labelled evaluation rather than restarting this scorecard.')
        if self.scorecard_id is None:
            self.scorecard_id = self.arcade.create_scorecard(tags=['ahr-v2-xl', 'public-interface'])
        env = self.arcade.make(str(game_id), seed=int(self.cfg.get('environment', {}).get('seed', 42)),
                               scorecard_id=self.scorecard_id, save_recording=False, render_mode=None)
        if env is None:
            raise RuntimeError(f'Official toolkit could not open {game_id}; verify accessible public assets and operation mode.')
        # make() initializes the official wrapper. Avoid an extra RESET action.
        initial = env.observation_space
        if initial is None:
            initial = env.reset()
        observation = public_observation(initial, str(game_id))
        self.env = env
        self.game_id = observation['game_id']
        self.episode_id = str(uuid.uuid4())
        self.actions_used = 0
        self._started_families.add(family)
        observation.update(episode_id=self.episode_id, actions_used=0)
        self.observation = observation
        return copy.deepcopy(observation)

    def step(self, action):
        from arcengine import GameAction
        if self.closed or self.env is None or self.observation is None:
            raise RuntimeError('Call reset(game_id) on an open session first.')
        action = action if isinstance(action, ActionSpec) else ActionSpec.from_dict(action)
        if self.actions_used >= self.max_actions:
            raise RuntimeError('Environment action budget exhausted; this is truncation, not a game failure proof.')
        if action.action != 0 and action.action not in self.observation['available_actions']:
            raise ValueError(f'Action {action.action} is absent from the latest public available_actions.')
        if terminal_observation(self.observation) and action.action != 0:
            raise RuntimeError('Episode is terminal; stop collection or explicitly RESET.')
        public_action = GameAction.RESET if action.action == 0 else getattr(GameAction, f'ACTION{action.action}')
        data = {'x': int(action.x), 'y': int(action.y)} if action.action == 6 else None
        raw = self.env.step(public_action, data=data)
        self.actions_used += 1
        # A failed response may have committed remotely. Never automatically retry it.
        observation = public_observation(raw, self.game_id)
        observation.update(episode_id=self.episode_id, actions_used=self.actions_used)
        self.observation = observation
        return copy.deepcopy(observation)

    def state_dict(self):
        return {'schema_version': 1, 'operation_mode': self.operation_mode,
                'toolkit_version': self.toolkit_version, 'game_id': self.game_id,
                'episode_id': self.episode_id, 'actions_used': self.actions_used,
                'scorecard_id': self.scorecard_id,
                'last_public_observation': None if self.observation is None else observation_to_record(self.observation),
                'exact_environment_resume_supported': False,
                'resume_policy': 'discard-interrupted-episode-and-start-new-id'}

    def resume(self, state, allow_restart=False):
        if not allow_restart:
            raise RuntimeError('Exact hidden environment restoration is unsupported by the verified public toolkit. Use allow_restart=True only to start a separately identified new episode.')
        if self.operation_mode == 'COMPETITION':
            raise RuntimeError('Competition episodes are not automatically replayed or resumed after a process crash.')
        if str(state.get('operation_mode', '')).upper() != self.operation_mode:
            raise ValueError('Resume operation mode mismatch.')
        return self.reset(state['game_id'])

    def close(self):
        if self.closed:
            return None
        self.closed = True
        return self.arcade.close_scorecard(self.scorecard_id) if self.scorecard_id is not None else None

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, traceback):
        self.close()
        return False

## Prepare public OFFLINE assets through the official SDK

The toolkit package does not bundle the public games. The verified 0.9.9 implementation of [`Arcade.make`](https://github.com/arcprize/ARC-AGI/blob/main/arc_agi/base.py) in **NORMAL** mode downloads and prepares an accessible public game under `environments_dir`; **OFFLINE** subsequently scans that directory. `acquire_public_assets` below uses only the public `list_games`/`reset` adapter operations and never inspects a game implementation, baseline-action metadata, or hidden state. The SDK's downloaded game files are third-party environment dependencies, not authored experiment helpers.

The following Run All setting `ACQUIRE_PUBLIC_ASSETS = True` prepares missing public assets over the network. Set it to False if assets are already supplied or you want to remain offline. This does not accept Kaggle terms, fetch its private data, submit, or train. A bounded report lists prepared game IDs and errors. Then select family-disjoint train/dev/heldout IDs in configuration; acquisition itself does not assign any training split.

In [ ]:
def acquire_public_assets(cfg,game_ids=None,max_games=32,max_wall_seconds=600):
    import os
    require_recovery_gate(root=ROOT)
    max_games,max_wall_seconds=int(max_games),float(max_wall_seconds)
    if not 1<=max_games<=32 or not 0<max_wall_seconds<=600:
        raise ValueError('Public preparation must be bounded to <=32 games and <=600 seconds.')
    override=os.environ.get('OPERATION_MODE','').strip().upper()
    if override not in {'','NORMAL'}:
        raise RuntimeError('Clear OPERATION_MODE before public NORMAL-mode preparation; the official SDK lets environment overrides change NORMAL.')
    asset_cfg=copy.deepcopy(cfg)
    asset_cfg['environment']['operation_mode']='NORMAL'
    asset_cfg['environment']['max_actions']=1
    started=time.monotonic()
    prepared,failures=[],[]
    with RunLock(root=ROOT,run_id='public-arc-assets'):
        with ARCSession(asset_cfg,operation_mode='NORMAL') as preparation:
            catalogue=preparation.list_games()
            chosen=list(game_ids) if game_ids is not None else catalogue[:max_games]
            if len(chosen)>max_games:
                raise ValueError('Requested public asset list exceeds bounded maximum.')
            families={game.split('-',1)[0] for game in catalogue}
            for game_id in chosen:
                if str(game_id).split('-',1)[0] not in families:
                    raise ValueError(f'{game_id} is absent from the public SDK catalogue.')
                if time.monotonic()-started>=max_wall_seconds:
                    break
                try:
                    observed=preparation.reset(str(game_id))
                    prepared.append(observed['game_id'])
                except Exception as exc:
                    failures.append({'game_id':str(game_id),'error_type':type(exc).__name__})
        with ARCSession(cfg,operation_mode='OFFLINE') as verification:
            local_games=verification.list_games()
        report={'schema_version':1,'status':'prepared' if prepared else 'no-assets-prepared',
                'public_catalogue':catalogue,'prepared':prepared,'offline_games':local_games,'failures':failures,
                'elapsed_seconds':time.monotonic()-started,'scope':'public-development-assets-only',
                'training_split_assigned':False,'source_inspected_by_project':False,
                'toolkit_version':importlib.metadata.version('arc-agi')}
        atomic_json(ROOT/'reports'/'public_assets.json',report)
    if not local_games:
        raise RuntimeError('No OFFLINE public assets are available. Check network/key accessibility or supply public environment_files; see reports/public_assets.json. No private data were fetched.')
    return report

### Later Run All: bounded public preparation
This cell may make network requests and let the official toolkit install public environment dependencies. It does not run training. Existing local catalogue entries are reused. Public game IDs are printed without secrets; choose configured families before the subsequent smoke/data cells.

In [ ]:
cfg=load_config()
require_recovery_gate(root=ROOT)
ACQUIRE_PUBLIC_ASSETS = True
with ARCSession(cfg,operation_mode='OFFLINE') as asset_check:
    existing_public_games=asset_check.list_games()
if not existing_public_games and ACQUIRE_PUBLIC_ASSETS:
    public_assets_report=acquire_public_assets(cfg,max_games=32,max_wall_seconds=600)
    existing_public_games=public_assets_report['offline_games']
print({'available_public_offline_game_ids':existing_public_games,'training_split_assigned':False})
if not existing_public_games:
    raise RuntimeError('No local public assets. Enable ACQUIRE_PUBLIC_ASSETS or supply official public environment_files.')

## Bounded Run All smoke
This execution cell is for the user's later Run All. It verifies the recovery gate first, discovers public games through the toolkit, and opens the configured game (or the first configured train game). It does not assign public games to training automatically. A missing game or empty catalogue stops clearly. The action is explicitly labelled a smoke action, never an expert demonstration.

In [ ]:
cfg = load_config()
require_recovery_gate(root=ROOT)
if str(cfg['environment'].get('operation_mode', 'OFFLINE')).upper() != 'OFFLINE':
    raise RuntimeError('Notebook 02 Run All smoke is OFFLINE only. Use ARCSession explicitly for a separately configured remote evaluation.')
smoke_started = time.monotonic()
with ARCSession(cfg, operation_mode='OFFLINE') as smoke_session:
    public_games = smoke_session.list_games()
    print({'public_game_count': len(public_games), 'public_game_ids': public_games[:40]})
    game = cfg['environment'].get('game_id') or next(iter(cfg['data'].get('train_games', [])), None)
    if game is None:
        raise RuntimeError('Choose environment.game_id or data.train_games in config/ahr_v2_xl.json after viewing the public catalogue. No automatic training split was created.')
    smoke_observation = smoke_session.reset(game)
    choices = [a for a in smoke_observation['available_actions'] if a != 0]
    smoke_step = None
    if choices and not terminal_observation(smoke_observation) and time.monotonic() - smoke_started < 30:
        smoke_action = ActionSpec(choices[0], x=0, y=0) if choices[0] == 6 else ActionSpec(choices[0])
        smoke_step = smoke_session.step(smoke_action)
    report = {'status': 'passed', 'kind': 'public-interface-smoke-not-performance', 'game_id': smoke_observation['game_id'],
              'toolkit_version': smoke_session.toolkit_version, 'action_count': smoke_session.actions_used,
              'initial_state': smoke_observation['state'], 'final_state': (smoke_step or smoke_observation)['state'],
              'exact_environment_resume_supported': False, 'elapsed_seconds': time.monotonic() - smoke_started}
atomic_json(ROOT / 'reports' / 'environment_smoke.json', report)
print(report)

## Recovery and benchmark boundaries
`state_dict()` saves public evidence and counters only. It does not include SDK internals, private fields, game objects, source code, or server cookies. A timeout after submitting an action leaves commitment uncertain; the adapter raises and the collector abandons that episode. Official completion and scorecard values come from the environment. Public development results must be labelled separately from private competition results.